In [ ]:
import torch
assert torch.cuda.is_available()
DEVICE = "cuda"
from google.colab import drive
drive.mount('/content/drive')
import os, glob, copy, time, random, shutil, json, zipfile, io
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from scipy.signal import find_peaks
from scipy.stats import norm
import matplotlib.pyplot as plt
from IPython.display import Image, display
import librosa
from tqdm import tqdm
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
OUT = "results"; os.makedirs(OUT, exist_ok=True)
SR, DUR, MAXLEN = 8000, 2.0, 16000
N_MELS, N_FFT, HOP = 64, 256, 102
CACHE = "physionet_cache.npz"
CACHE_S1S2 = "physionet_cache_s1s2.npz"
DRIVE_DIR = "/content/drive/MyDrive"
def find_zip(name):
    p = os.path.join(DRIVE_DIR, f"{name}.zip")
    if os.path.isfile(p): return p
    for root, _, files in os.walk(DRIVE_DIR):
        for f in files:
            if f.lower() == f"{name}.zip".lower():
                return os.path.join(root, f)
    return None
TRAIN_ZIP = find_zip("training")
VAL_ZIP = find_zip("validation")
if TRAIN_ZIP and not glob.glob("data/training-*/REFERENCE.csv"):
    os.makedirs("data", exist_ok=True)
    with zipfile.ZipFile(TRAIN_ZIP) as z: z.extractall("data")
if VAL_ZIP and not glob.glob("data/validation*/REFERENCE.csv"):
    with zipfile.ZipFile(VAL_ZIP) as z: z.extractall("data")
def collect_train():
    wavs = []
    for csv in glob.glob("data/training-*/REFERENCE.csv"):
        folder = os.path.dirname(csv)
        with open(csv) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                name, lab = line.split(",")[:2]
                lab = 0 if int(lab) == -1 else 1
                wp = os.path.join(folder, name + ".wav")
                if os.path.isfile(wp): wavs.append((wp, lab))
    return wavs
def collect_val():
    wavs = []
    for pat in ["data/validation*/REFERENCE.csv", "data/validation/*/REFERENCE.csv"]:
        for csv in glob.glob(pat):
            folder = os.path.dirname(csv)
            with open(csv) as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    p = line.split(",")
                    if len(p) < 2: continue
                    name, lab = p[0], p[1]
                    try: lab = int(lab)
                    except: continue
                    lab = 0 if lab == -1 else 1
                    for cand in [os.path.join(folder, name + ".wav"),
                                 os.path.join(folder, name.split(".")[0] + ".wav")]:
                        if os.path.isfile(cand):
                            wavs.append((cand, lab)); break
        if wavs: break
    return wavs
all_wavs = collect_train()
val_wavs = collect_val()
random.seed(SEED); random.shuffle(all_wavs)
cut = int(0.8 * len(all_wavs))
tr_list, te_list = all_wavs[:cut], all_wavs[cut:]
print(f"Train: {len(tr_list)}  Test: {len(te_list)}  Val: {len(val_wavs)}")
def load_wave(p):
    y, _ = librosa.load(p, sr=SR)
    return y[:MAXLEN] if len(y) > MAXLEN else np.pad(y, (0, MAXLEN - len(y)))
def wav_to_mel(w, n_mels=N_MELS, n_fft=N_FFT, hop=HOP):
    S = librosa.feature.melspectrogram(y=w, sr=SR, n_mels=n_mels, n_fft=n_fft, hop_length=hop)
    return librosa.power_to_db(S, ref=np.max)
def build_features(lst, desc, n_mels=N_MELS, n_fft=N_FFT, hop=HOP):
    X, Y = [], []
    for wp, lb in tqdm(lst, desc=desc, leave=False):
        mel = wav_to_mel(load_wave(wp), n_mels, n_fft, hop)
        if mel.shape[1] < 157: mel = np.pad(mel, ((0, 0), (0, 157 - mel.shape[1])))
        X.append(mel[:, :157]); Y.append(lb)
    return (np.array(X)[:, None, :, :].astype(np.float32),
            np.array(Y).astype(np.int64))
if os.path.isfile(CACHE):
    _z = np.load(CACHE)
    X_train, y_train = _z["X_train"], _z["y_train"]
    X_test,  y_test  = _z["X_test"],  _z["y_test"]
else:
    X_train, y_train = build_features(tr_list, "train")
    X_test,  y_test  = build_features(te_list, "test")
    np.savez_compressed(CACHE, X_train=X_train, y_train=y_train,
                        X_test=X_test, y_test=y_test)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Majority baseline: {np.bincount(y_test).max()/len(y_test)*100:.2f}%")
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, i, o, k=3, s=1):
        super().__init__()
        self.dw = nn.Conv2d(i, i, k, s, padding=k//2, groups=i)
        self.pw = nn.Conv2d(i, o, 1)
        self.bn = nn.BatchNorm2d(o); self.relu = nn.ReLU()
    def forward(self, x): return self.relu(self.bn(self.pw(self.dw(x))))
class SEBlock(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.sq = nn.AdaptiveAvgPool2d(1)
        self.ex = nn.Sequential(nn.Linear(c, c//r), nn.ReLU(),
                                nn.Linear(c//r, c), nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.shape
        y = self.sq(x).view(b, c)
        return x * self.ex(y).view(b, c, 1, 1)
class EfficientHeartNet(nn.Module):
    def __init__(self, n=2, drop=0.2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(2),
            DepthwiseSeparableConv(16, 32), SEBlock(32), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64*4*4, 32), nn.ReLU(),
            nn.Dropout(drop), nn.Linear(32, n))
    def forward(self, x): return self.classifier(self.features(x))
print(f"Model params: {sum(p.numel() for p in EfficientHeartNet().parameters())}")
def add_noise(x, snr_db, seed=None):
    if seed is not None: torch.manual_seed(seed)
    sp = (x ** 2).mean()
    return x + torch.randn_like(x) * ((sp / (10 ** (snr_db / 10))) ** 0.5)
def freeze_bn(m):
    for mod in m.modules():
        if isinstance(mod, (nn.BatchNorm1d, nn.BatchNorm2d)): mod.eval()
def evaluate(model, X, y, bs=64):
    model.eval()
    dl = DataLoader(TensorDataset(torch.tensor(X).float(),
                                  torch.tensor(y).long()), batch_size=bs)
    ps, ts, prs = [], [], []
    with torch.no_grad():
        for xb, yb in dl:
            out = model(xb.to(DEVICE))
            ps.append(out.argmax(1).cpu().numpy())
            prs.append(F.softmax(out, dim=1)[:, 1].cpu().numpy())
            ts.append(yb.numpy())
    p = np.concatenate(ps); t = np.concatenate(ts); pr = np.concatenate(prs)
    try: auc = roc_auc_score(t, pr)
    except: auc = float("nan")
    return accuracy_score(t, p), auc
def train_warm(Xtr, ytr, Xte, yte, epochs=40, bs=32):
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    tr = DataLoader(TensorDataset(torch.tensor(Xtr).float(),
                                  torch.tensor(ytr).long()),
                    batch_size=bs, shuffle=True)
    net = EfficientHeartNet().to(DEVICE)
    opt = torch.optim.SGD(net.parameters(), lr=1e-2, momentum=0.9,
                          weight_decay=5e-4, nesterov=True)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    crit = nn.CrossEntropyLoss()
    best_val, best_state = 0.0, None
    for ep in range(epochs):
        net.train()
        for xb, yb in tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(net(xb), yb).backward(); opt.step()
        sch.step()
        if (ep + 1) % 10 == 0:
            va, _ = evaluate(net, Xte, yte)
            if va > best_val:
                best_val = va; best_state = copy.deepcopy(net.state_dict())
    net.load_state_dict(best_state)
    return net, best_val
print("\n=== Warm-start ===")
warm_net, warm_val = train_warm(X_train, y_train, X_test, y_test)
WARM_STATE = copy.deepcopy(warm_net.state_dict())
print(f"Warm-start best val: {warm_val*100:.2f}%")
def dirichlet(labels, nc, alpha, seed=SEED):
    rng = np.random.RandomState(seed)
    labels = np.asarray(labels)
    ci = [[] for _ in range(nc)]
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]; rng.shuffle(idx)
        p = rng.dirichlet([alpha]*nc); p = np.maximum(p, 1e-3); p /= p.sum()
        sp = (np.cumsum(p)*len(idx)).astype(int)[:-1]
        for i, ch in enumerate(np.split(idx, sp)):
            ci[i].extend(ch.tolist())
    for i in range(nc):
        if len(ci[i]) < 10:
            ci[i].extend(rng.choice(len(labels), 10 - len(ci[i]),
                                    replace=False).tolist())
    return [np.array(x) for x in ci]
def make_client_loaders(X, y, nc, alpha, bs=16):
    idx = dirichlet(y, nc, alpha)
    loaders = [DataLoader(TensorDataset(torch.tensor(X[ci]).float(),
                                         torch.tensor(y[ci]).long()),
                          batch_size=min(bs, len(ci)), shuffle=True) for ci in idx]
    return loaders, [len(ci) for ci in idx]
def run_fedavg(Xtr, ytr, Xte, yte, warm_state, nc=10, alpha=0.5, E=2,
               rounds=20, lr=1e-3, freeze=True):
    torch.manual_seed(SEED); np.random.seed(SEED)
    loaders, sizes = make_client_loaders(Xtr, ytr, nc, alpha)
    gnet = EfficientHeartNet().to(DEVICE); gnet.load_state_dict(warm_state)
    clients = [copy.deepcopy(gnet) for _ in range(nc)]
    hist, drift = [], {c: [] for c in range(nc)}
    for r in range(rounds):
        for c in range(nc):
            m = clients[c]; m.train()
            if freeze: freeze_bn(m)
            opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
            for _ in range(E):
                for xb, yb in loaders[c]:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    opt.zero_grad()
                    nn.CrossEntropyLoss()(m(xb), yb).backward()
                    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                    opt.step()
        with torch.no_grad():
            gd = gnet.state_dict(); tot = float(sum(sizes))
            w = [s/tot for s in sizes]
            for k in gd:
                if gd[k].dtype in (torch.float32, torch.float64):
                    gd[k] = torch.stack([clients[c].state_dict()[k].float()*w[c]
                                         for c in range(nc)], 0).sum(0)
                else:
                    gd[k] = clients[0].state_dict()[k]
            for c in range(nc):
                s = 0.0
                for k in gd:
                    if gd[k].dtype in (torch.float32, torch.float64):
                        s += (clients[c].state_dict()[k].float()-gd[k]).pow(2).sum().item()
                drift[c].append(float(np.sqrt(s)))
            gnet.load_state_dict(gd)
            for c in range(nc): clients[c].load_state_dict(gd)
        hist.append(evaluate(gnet, Xte, yte)[0])
    return hist, drift, gnet
def run_fedprox(Xtr, ytr, Xte, yte, warm_state, mu=0.01, nc=10, alpha=0.5,
                E=2, rounds=20, lr=1e-3):
    torch.manual_seed(SEED); np.random.seed(SEED)
    loaders, sizes = make_client_loaders(Xtr, ytr, nc, alpha)
    gnet = EfficientHeartNet().to(DEVICE); gnet.load_state_dict(warm_state)
    clients = [copy.deepcopy(gnet) for _ in range(nc)]
    hist, drift = [], {c: [] for c in range(nc)}
    for r in range(rounds):
        gp = {k: v.detach().clone() for k, v in gnet.state_dict().items()
              if v.dtype in (torch.float32, torch.float64)}
        for c in range(nc):
            m = clients[c]; m.train(); freeze_bn(m)
            opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
            for _ in range(E):
                for xb, yb in loaders[c]:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    opt.zero_grad()
                    loss = nn.CrossEntropyLoss()(m(xb), yb)
                    prox = sum((p - gp[n]).pow(2).sum()
                               for n, p in m.named_parameters()
                               if n in gp and p.requires_grad)
                    (loss + (mu/2.0)*prox).backward()
                    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                    opt.step()
        with torch.no_grad():
            gd = gnet.state_dict(); tot = float(sum(sizes))
            w = [s/tot for s in sizes]
            for k in gd:
                if gd[k].dtype in (torch.float32, torch.float64):
                    gd[k] = torch.stack([clients[c].state_dict()[k].float()*w[c]
                                         for c in range(nc)], 0).sum(0)
                else:
                    gd[k] = clients[0].state_dict()[k]
            for c in range(nc):
                s = 0.0
                for k in gd:
                    if gd[k].dtype in (torch.float32, torch.float64):
                        s += (clients[c].state_dict()[k].float()-gd[k]).pow(2).sum().item()
                drift[c].append(float(np.sqrt(s)))
            gnet.load_state_dict(gd)
            for c in range(nc): clients[c].load_state_dict(gd)
        hist.append(evaluate(gnet, Xte, yte)[0])
    return hist, drift, gnet
def run_fedbn(Xtr, ytr, Xte, yte, warm_state, nc=10, alpha=0.5, E=2,
              rounds=20, lr=1e-3):
    torch.manual_seed(SEED); np.random.seed(SEED)
    loaders, sizes = make_client_loaders(Xtr, ytr, nc, alpha)
    gnet = EfficientHeartNet().to(DEVICE); gnet.load_state_dict(warm_state)
    clients = [copy.deepcopy(gnet) for _ in range(nc)]
    bn_keys = set()
    for nm, mod in gnet.named_modules():
        if isinstance(mod, (nn.BatchNorm1d, nn.BatchNorm2d)):
            for pk in mod.state_dict().keys():
                bn_keys.add(f"{nm}.{pk}" if nm else pk)
    hist, drift = [], {c: [] for c in range(nc)}
    for r in range(rounds):
        for c in range(nc):
            m = clients[c]; m.train()
            opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
            for _ in range(E):
                for xb, yb in loaders[c]:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    opt.zero_grad()
                    nn.CrossEntropyLoss()(m(xb), yb).backward()
                    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                    opt.step()
        with torch.no_grad():
            gd = gnet.state_dict(); tot = float(sum(sizes))
            w = [s/tot for s in sizes]
            for k in gd:
                if k in bn_keys: continue
                if gd[k].dtype in (torch.float32, torch.float64):
                    gd[k] = torch.stack([clients[c].state_dict()[k].float()*w[c]
                                         for c in range(nc)], 0).sum(0)
                else:
                    gd[k] = clients[0].state_dict()[k]
            for c in range(nc):
                s = 0.0
                for k in gd:
                    if k in bn_keys: continue
                    if gd[k].dtype in (torch.float32, torch.float64):
                        s += (clients[c].state_dict()[k].float()-gd[k]).pow(2).sum().item()
                drift[c].append(float(np.sqrt(s)))
            gnet.load_state_dict(gd, strict=False)
            for c in range(nc):
                st = clients[c].state_dict()
                for k in gd:
                    if k not in bn_keys: st[k] = gd[k]
                clients[c].load_state_dict(st)
        hist.append(evaluate(gnet, Xte, yte)[0])
    return hist, drift, gnet
print("\n=== Batch 1 ===")
ALPHAS = [0.3, 0.5, 0.7, 0.9]
alpha_rows = []
for a in ALPHAS:
    h, dr, _ = run_fedavg(X_train, y_train, X_test, y_test, WARM_STATE,
                          nc=10, alpha=a, E=2, rounds=20)
    alpha_rows.append({"alpha": a, "Final Acc (%)": f"{h[-1]*100:.2f}",
                       "Avg Client Drift": f"{np.mean([np.mean(v[-5:]) for v in dr.values()]):.4f}"})
df_alpha = pd.DataFrame(alpha_rows)
print(df_alpha.to_string(index=False))
df_alpha.to_csv(f"{OUT}/batch1_alpha.csv", index=False)
E_LIST = [1, 2, 4, 8]
epoch_rows = []
for E in E_LIST:
    h, dr, _ = run_fedavg(X_train, y_train, X_test, y_test, WARM_STATE,
                          nc=10, alpha=0.5, E=E, rounds=20)
    epoch_rows.append({"E": E, "Final Acc (%)": f"{h[-1]*100:.2f}",
                       "Avg Client Drift": f"{np.mean([np.mean(v[-5:]) for v in dr.values()]):.4f}"})
df_epoch = pd.DataFrame(epoch_rows)
print(df_epoch.to_string(index=False))
df_epoch.to_csv(f"{OUT}/batch1_epochs.csv", index=False)
h, dr, _ = run_fedavg(X_train, y_train, X_test, y_test, WARM_STATE,
                      nc=10, alpha=0.5, E=2, rounds=20)
final_d = [dr[c][-1] for c in range(10)]
_, sizes_10 = make_client_loaders(X_train, y_train, 10, 0.5)
df_drift = pd.DataFrame({"Client": list(range(10)), "Samples": sizes_10,
                         "Final L2 Drift": [f"{d:.4f}" for d in final_d]})
print(df_drift.to_string(index=False))
df_drift.to_csv(f"{OUT}/batch1_drift.csv", index=False)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].bar(range(len(ALPHAS)), [float(r["Final Acc (%)"]) for r in alpha_rows], color='steelblue')
ax[0].set_xticks(range(len(ALPHAS))); ax[0].set_xticklabels([f"alpha={a}" for a in ALPHAS])
ax[0].set_ylabel("Final Acc (%)"); ax[0].set_title("(a) alpha-sweep"); ax[0].grid(axis='y', alpha=0.3)
ax[1].bar(range(len(E_LIST)), [float(r["Final Acc (%)"]) for r in epoch_rows], color='salmon')
ax[1].set_xticks(range(len(E_LIST))); ax[1].set_xticklabels([f"E={E}" for E in E_LIST])
ax[1].set_ylabel("Final Acc (%)"); ax[1].set_title("(b) E-sweep"); ax[1].grid(axis='y', alpha=0.3)
ax[2].bar(range(10), final_d, color='mediumseagreen')
ax[2].set_xlabel("Client"); ax[2].set_ylabel("L2 Drift")
ax[2].set_title("(c) Per-client drift"); ax[2].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT}/batch1_figure.png", dpi=200); plt.close()
display(Image(filename=f"{OUT}/batch1_figure.png"))
print("\n=== Batch 2 ===")
MEL_CONFIGS = [("32/256/102",32,256,102), ("64/256/102*",64,256,102),
               ("128/256/102",128,256,102), ("64/512/102",64,512,102),
               ("64/256/51",64,256,51)]
mel_rows = []
for name, nm, nf, hp in MEL_CONFIGS:
    Xtr, ytr = build_features(tr_list, f"mel {name}", nm, nf, hp)
    Xte, yte = build_features(te_list, f"mel {name}", nm, nf, hp)
    _, val = train_warm(Xtr, ytr, Xte, yte, epochs=30)
    mel_rows.append({"Config": name, "Val Acc (%)": f"{val*100:.2f}"})
df_mel = pd.DataFrame(mel_rows)
print(df_mel.to_string(index=False))
df_mel.to_csv(f"{OUT}/batch2_mel.csv", index=False)
SNRS = [20, 15, 10, 5]
def eval_snr(model, X, y, snr, bs=64):
    Xn = add_noise(torch.tensor(X).float(), snr).numpy()
    return evaluate(model, Xn, y, bs)[0]
cnn_clean = evaluate(warm_net, X_test, y_test)[0]
cnn_snr = [eval_snr(warm_net, X_test, y_test, s) for s in SNRS]
fl_net_ab = run_fedavg(X_train, y_train, X_test, y_test, WARM_STATE, nc=10, alpha=0.9, E=2, rounds=20)[2]
fl_clean = evaluate(fl_net_ab, X_test, y_test)[0]
fl_snr = [eval_snr(fl_net_ab, X_test, y_test, s) for s in SNRS]
df_ab = pd.DataFrame([
    {"Config":"CNN","Clean":f"{cnn_clean*100:.2f}",
     **{f"{s}dB":f"{a*100:.2f}" for s,a in zip(SNRS,cnn_snr)}},
    {"Config":"CNN+FL","Clean":f"{fl_clean*100:.2f}",
     **{f"{s}dB":f"{a*100:.2f}" for s,a in zip(SNRS,fl_snr)}},
])
print(df_ab.to_string(index=False))
df_ab.to_csv(f"{OUT}/batch2_ablation.csv", index=False)
def quantize(model, bits):
    q = copy.deepcopy(model).cpu()
    mi = (2 ** (bits - 1)) - 1
    for m in q.modules():
        if isinstance(m, nn.Conv2d):
            w = m.weight.data
            wm = w.view(w.size(0), -1).abs().max(1)[0].clamp(min=1e-8)
            sc = (wm / mi).view(-1, 1, 1, 1)
            m.weight.data = torch.round(w / sc) * sc
        elif isinstance(m, nn.Linear):
            w = m.weight.data
            wm = w.abs().max(1, keepdim=True)[0].clamp(min=1e-8)
            sc = wm / mi
            m.weight.data = torch.round(w / sc) * sc
    return q
def per_class(model, X, y):
    model.eval().to(DEVICE)
    dl = DataLoader(TensorDataset(torch.tensor(X).float(),
                                  torch.tensor(y).long()), batch_size=64)
    ps, ts = [], []
    with torch.no_grad():
        for xb, yb in dl:
            ps.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
            ts.append(yb.numpy())
    p = np.concatenate(ps); t = np.concatenate(ts)
    return (accuracy_score(t[t==0], p[t==0]), accuracy_score(t[t==1], p[t==1]),
            accuracy_score(t, p))
fp32_n, fp32_a, fp32_o = per_class(copy.deepcopy(warm_net), X_test, y_test)
i8_n, i8_a, i8_o = per_class(quantize(warm_net, 8), X_test, y_test)
i4_n, i4_a, i4_o = per_class(quantize(warm_net, 4), X_test, y_test)
df_q = pd.DataFrame([
    {"Precision":"FP32","Normal":f"{fp32_n*100:.2f}","Abnormal":f"{fp32_a*100:.2f}","Overall":f"{fp32_o*100:.2f}"},
    {"Precision":"INT8","Normal":f"{i8_n*100:.2f}","Abnormal":f"{i8_a*100:.2f}","Overall":f"{i8_o*100:.2f}"},
    {"Precision":"INT4","Normal":f"{i4_n*100:.2f}","Abnormal":f"{i4_a*100:.2f}","Overall":f"{i4_o*100:.2f}"},
])
print(df_q.to_string(index=False))
df_q.to_csv(f"{OUT}/batch2_quant.csv", index=False)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].bar(range(len(mel_rows)), [float(r["Val Acc (%)"]) for r in mel_rows], color='steelblue')
ax[0].set_xticks(range(len(mel_rows))); ax[0].set_xticklabels([r["Config"] for r in mel_rows], rotation=20, ha='right', fontsize=8)
ax[0].set_ylabel("Val Acc (%)"); ax[0].set_title("(a) Mel sweep"); ax[0].grid(axis='y', alpha=0.3)
x = np.arange(2); w = 0.4
ax[1].bar(x-w/2, [cnn_clean*100, fl_clean*100], w, label='Clean', color='steelblue')
ax[1].bar(x+w/2, [cnn_snr[-1]*100, fl_snr[-1]*100], w, label='5 dB', color='salmon')
ax[1].set_xticks(x); ax[1].set_xticklabels(["CNN","CNN+FL"]); ax[1].legend()
ax[1].set_ylabel("Accuracy (%)"); ax[1].set_title("(b) Module ablation"); ax[1].grid(axis='y', alpha=0.3)
prec = ["FP32","INT8","INT4"]
ax[2].bar(np.arange(3)-0.27, [fp32_n*100,i8_n*100,i4_n*100], 0.27, label='Normal', color='steelblue')
ax[2].bar(np.arange(3),     [fp32_a*100,i8_a*100,i4_a*100], 0.27, label='Abnormal', color='salmon')
ax[2].bar(np.arange(3)+0.27,[fp32_o*100,i8_o*100,i4_o*100], 0.27, label='Overall', color='gray')
ax[2].set_xticks(range(3)); ax[2].set_xticklabels(prec); ax[2].legend(fontsize=8)
ax[2].set_ylabel("Accuracy (%)"); ax[2].set_title("(c) Quantization"); ax[2].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT}/batch2_figure.png", dpi=200); plt.close()
display(Image(filename=f"{OUT}/batch2_figure.png"))
print("\n=== Batch 3 ===")
X_all = np.concatenate([X_train, X_test], 0)
y_all = np.concatenate([y_train, y_test], 0)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(X_all, y_all))
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=1.0):
        super().__init__()
        if weight is not None: self.register_buffer("weight", weight)
        else: self.weight = None
        self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()
class CBLoss(nn.Module):
    def __init__(self, counts, beta=0.999):
        super().__init__()
        eff = 1.0 - np.power(beta, counts)
        w = (1.0 - beta) / eff
        w = w / w.sum() * len(counts)
        self.register_buffer("weights", torch.tensor(w, dtype=torch.float32))
    def forward(self, logits, targets):
        return F.cross_entropy(logits, targets, weight=self.weights.to(logits.device))
def train_cv(Xtr, ytr, Xva, yva, loss_fn, epochs=30, bs=32):
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    loss_fn = loss_fn.to(DEVICE)
    tr = DataLoader(TensorDataset(torch.tensor(Xtr).float(), torch.tensor(ytr).long()),
                    batch_size=bs, shuffle=True)
    net = EfficientHeartNet().to(DEVICE)
    opt = torch.optim.SGD(net.parameters(), lr=1e-2, momentum=0.9, weight_decay=5e-4, nesterov=True)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    best_val, best_state = 0.0, None
    for ep in range(epochs):
        net.train()
        for xb, yb in tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss_fn(net(xb), yb).backward(); opt.step()
        sch.step()
        if (ep+1) % 5 == 0:
            va = evaluate(net, Xva, yva)[0]
            if va > best_val:
                best_val = va; best_state = copy.deepcopy(net.state_dict())
    if best_state: net.load_state_dict(best_state)
    return evaluate(net, Xva, yva)
cv_rows = []
for i, (tr_idx, va_idx) in enumerate(folds, 1):
    acc, auc = train_cv(X_all[tr_idx], y_all[tr_idx], X_all[va_idx], y_all[va_idx],
                        nn.CrossEntropyLoss())
    cv_rows.append({"Fold": i, "Acc": f"{acc*100:.2f}", "AUC": f"{auc:.4f}"})
df_cv = pd.DataFrame(cv_rows)
print(df_cv.to_string(index=False))
df_cv.to_csv(f"{OUT}/batch3_cv.csv", index=False)
def make_loss(name, ytr):
    counts = np.maximum(np.bincount(ytr, minlength=2).astype(np.float32), 1)
    if name == "CE": return nn.CrossEntropyLoss()
    if name == "WCE":
        w = 1.0 / np.sqrt(counts); w = w / w.sum() * len(counts)
        return nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32))
    if name == "Focal": return FocalLoss(gamma=1.0)
    if name == "CB": return CBLoss(counts.tolist())
loss_rows = []
for lname in ["CE", "WCE", "Focal", "CB"]:
    per = []
    for tr_idx, va_idx in folds:
        acc, auc = train_cv(X_all[tr_idx], y_all[tr_idx],
                            X_all[va_idx], y_all[va_idx],
                            make_loss(lname, y_all[tr_idx]))
        per.append((acc, auc))
    loss_rows.append({"Loss": lname,
                      "Acc (%)": f"{np.mean([p[0] for p in per])*100:.2f} +/- {np.std([p[0] for p in per])*100:.2f}",
                      "AUC": f"{np.mean([p[1] for p in per]):.4f}"})
df_loss = pd.DataFrame(loss_rows)
print(df_loss.to_string(index=False))
df_loss.to_csv(f"{OUT}/batch3_loss.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].bar([r["Fold"] for r in cv_rows], [float(r["Acc"]) for r in cv_rows], color='steelblue')
ax[0].axhline(np.mean([float(r["Acc"]) for r in cv_rows]), ls='--', color='red', alpha=0.7)
ax[0].set_xlabel("Fold"); ax[0].set_ylabel("Acc (%)"); ax[0].set_title("(a) 5-fold CV"); ax[0].grid(axis='y', alpha=0.3)
ax[1].bar(range(4), [float(r["Acc (%)"].split()[0]) for r in loss_rows], color='salmon')
ax[1].set_xticks(range(4)); ax[1].set_xticklabels([r["Loss"] for r in loss_rows])
ax[1].set_ylabel("Acc (%)"); ax[1].set_title("(b) Loss ablation"); ax[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT}/batch3_figure.png", dpi=200); plt.close()
display(Image(filename=f"{OUT}/batch3_figure.png"))
print("\n=== Batch 4 ===")
fl_results = {}
for name, fn in [("FedAvg", run_fedavg), ("FedProx", run_fedprox), ("FedBN", run_fedbn)]:
    h, dr, _ = fn(X_train, y_train, X_test, y_test, WARM_STATE, nc=10, alpha=0.5, rounds=20)
    fl_results[name] = {
        "Final": f"{h[-1]*100:.2f}",
        "Drift": f"{np.mean([np.mean(v[-5:]) for v in dr.values()]):.4f}",
        "_acc": h[-1], "_hist": h,
    }
df_fl = pd.DataFrame([{"Algorithm": k, "Final Acc (%)": v["Final"],
                       "Avg Drift": v["Drift"]} for k, v in fl_results.items()])
print(df_fl.to_string(index=False))
df_fl.to_csv(f"{OUT}/batch4_fl.csv", index=False)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for k, v in fl_results.items():
    ax[0].plot(range(1, 21), [x*100 for x in v["_hist"]], marker='o', label=k)
ax[0].axhline(warm_val*100, ls='--', color='grey', alpha=0.6, label='Centralized')
ax[0].set_xlabel("Round"); ax[0].set_ylabel("Acc (%)"); ax[0].legend(fontsize=8)
ax[0].set_title("(a) Convergence"); ax[0].grid(alpha=0.3)
ax[1].bar(range(3), [v["_acc"]*100 for v in fl_results.values()], color=['steelblue','salmon','mediumseagreen'])
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(list(fl_results.keys()))
ax[1].set_ylabel("Final Acc (%)"); ax[1].set_title("(b) Final accuracy"); ax[1].grid(axis='y', alpha=0.3)
ax[2].bar(range(3), [float(v["Drift"]) for v in fl_results.values()], color=['steelblue','salmon','mediumseagreen'])
ax[2].set_xticks(range(3)); ax[2].set_xticklabels(list(fl_results.keys()))
ax[2].set_ylabel("Drift"); ax[2].set_title("(c) Client drift"); ax[2].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT}/batch4_figure.png", dpi=200); plt.close()
display(Image(filename=f"{OUT}/batch4_figure.png"))
print("\n=== Batch 5 ===")
def bayesian_recal(probs, snr, T_max=2.5, max_shift=0.25, ct=0.75):
    a = 1.0 / (1.0 + np.exp((snr - 15) / 5))
    T = 1.0 + (T_max - 1.0) * a
    mc = probs.max(1).values
    u = torch.clamp((ct - mc) / ct, 0, 1)
    pr_t = F.softmax(torch.log(probs.clamp_min(1e-9)) / T, dim=1)
    sh = (max_shift * a) * u
    prior = torch.cat([(0.5 - sh).unsqueeze(1), (0.5 + sh).unsqueeze(1)], 1)
    return (pr_t * prior) / (pr_t * prior).sum(1, keepdim=True)
def eval_thresh(model, X, y, snr, T=None):
    torch.manual_seed(SEED)
    Xn = add_noise(torch.tensor(X).float(), snr).numpy()
    dl = DataLoader(TensorDataset(torch.tensor(Xn).float(), torch.tensor(y).long()), 64)
    model.eval(); ps, ts, cs = [], [], []
    fire = T is not None and snr <= T
    with torch.no_grad():
        for xb, yb in dl:
            pr = F.softmax(model(xb.to(DEVICE)), 1)
            if fire: pr = bayesian_recal(pr, snr)
            ps.append(pr.argmax(1).cpu().numpy())
            cs.append(pr.max(1).values.cpu().numpy())
            ts.append(yb.numpy())
    return accuracy_score(np.concatenate(ts), np.concatenate(ps)), np.concatenate(cs), np.concatenate(ts), np.concatenate(ps)
TEST_SNRS = [30, 25, 20, 15, 10, 5]
THRESH = [10, 15, 20, 25, 30]
base_acc = {s: eval_thresh(warm_net, X_test, y_test, s)[0] for s in TEST_SNRS}
thresh_rows = []
for T in THRESH:
    row = {"Threshold": T}
    for s in TEST_SNRS:
        a, *_ = eval_thresh(warm_net, X_test, y_test, s, T=T)
        row[f"{s}dB"] = f"{a*100:.2f}"
    thresh_rows.append(row)
df_thresh = pd.DataFrame(thresh_rows)
print(df_thresh.to_string(index=False))
df_thresh.to_csv(f"{OUT}/batch5_threshold.csv", index=False)
def ece(confs, correct, n=10):
    edges = np.linspace(0, 1, n+1); e = 0.0
    for i in range(n):
        m = (confs > edges[i]) & (confs <= edges[i+1])
        if m.sum() == 0: continue
        e += m.sum()/len(confs) * abs(correct[m].mean() - confs[m].mean())
    return e
ece_rows = []
for s in TEST_SNRS:
    _, c1, t1, p1 = eval_thresh(warm_net, X_test, y_test, s)
    _, c2, t2, p2 = eval_thresh(warm_net, X_test, y_test, s, T=20)
    ece_rows.append({"SNR": s, "ECE no-recal": f"{ece(c1,(p1==t1).astype(int)):.4f}",
                     "ECE +recal": f"{ece(c2,(p2==t2).astype(int)):.4f}"})
df_ece = pd.DataFrame(ece_rows)
print(df_ece.to_string(index=False))
df_ece.to_csv(f"{OUT}/batch5_ece.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(TEST_SNRS, [base_acc[s]*100 for s in TEST_SNRS], 'k--o', label='No recal')
for i, T in enumerate(THRESH):
    y = [float(thresh_rows[i][f"{s}dB"]) for s in TEST_SNRS]
    ax[0].plot(TEST_SNRS, y, marker='o', label=f'T={T}')
ax[0].set_xlabel("SNR (dB)"); ax[0].set_ylabel("Acc (%)"); ax[0].legend(fontsize=8)
ax[0].set_title("(a) Threshold sweep"); ax[0].invert_xaxis(); ax[0].grid(alpha=0.3)
delta = np.zeros((5, 6))
for i, T in enumerate(THRESH):
    for j, s in enumerate(TEST_SNRS):
        delta[i, j] = (float(thresh_rows[i][f"{s}dB"]) - base_acc[s]*100)
im = ax[1].imshow(delta, cmap='RdYlGn', aspect='auto', vmin=-2, vmax=2)
ax[1].set_xticks(range(6)); ax[1].set_xticklabels([f"{s}" for s in TEST_SNRS])
ax[1].set_yticks(range(5)); ax[1].set_yticklabels([f"T={T}" for T in THRESH])
ax[1].set_title("(b) Delta Accuracy (%)")
for i in range(5):
    for j in range(6):
        ax[1].text(j, i, f"{delta[i,j]:+.2f}", ha='center', fontsize=8)
plt.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.savefig(f"{OUT}/batch5_figure.png", dpi=200); plt.close()
display(Image(filename=f"{OUT}/batch5_figure.png"))
print("\n=== Batch 6 ===")
def priv_eps(q, sigma, T, delta=1e-5):
    mu = q * np.sqrt(T) / sigma
    for e in np.linspace(0.1, 100, 500):
        d = norm.cdf(-e/mu + mu/2) - np.exp(e) * norm.cdf(-e/mu - mu/2)
        if d <= delta: return float(e)
    return 100.0
def dp_sgd(Xtr, ytr, Xte, yte, eps=None, sigma=None, epochs=20, bs=64, lr=5e-3, C=1.0):
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    N = len(Xtr); q = bs / N; T = epochs * (N // bs)
    if sigma is None and eps is not None:
        lo, hi = 0.1, 100.0
        for _ in range(40):
            m = (lo + hi) / 2
            (lo, hi) = (m, hi) if priv_eps(q, m, T) > eps else (lo, m)
        sigma = (lo + hi) / 2
    tr = DataLoader(TensorDataset(torch.tensor(Xtr).float(), torch.tensor(ytr).long()),
                    bs, shuffle=True)
    net = EfficientHeartNet().to(DEVICE)
    opt = torch.optim.SGD(net.parameters(), lr=lr)
    params = [p for p in net.parameters()]
    tot_n = sum(p.numel() for p in params)
    for ep in range(epochs):
        net.train()
        for xb, yb in tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            B = xb.size(0)
            opt.zero_grad()
            losses = nn.CrossEntropyLoss(reduction='none')(net(xb), yb)
            pg = torch.zeros(B, tot_n, device=DEVICE)
            for i in range(B):
                for p in params: p.grad = None
                losses[i].backward(retain_graph=True)
                off = 0
                for p in params:
                    g = p.grad if p.grad is not None else torch.zeros_like(p)
                    pg[i, off:off+p.numel()] = g.flatten(); off += p.numel()
            norms = pg.norm(dim=1, keepdim=True).clamp_min(1e-6)
            pg = pg * (C / norms).clamp(max=1.0)
            avg = pg.mean(0) + torch.randn(tot_n, device=DEVICE) * (sigma * C / B)
            with torch.no_grad():
                off = 0
                for p in params:
                    n = p.numel()
                    p.grad = avg[off:off+n].view_as(p).clone(); off += n
            opt.step()
    acc, auc = evaluate(net, Xte, yte)
    return net, acc, auc, sigma
base_acc_6, base_auc_6 = evaluate(warm_net, X_test, y_test)
dp_rows = [{"epsilon": "inf", "Acc (%)": f"{base_acc_6*100:.2f}", "AUC": f"{base_auc_6:.4f}"}]
for eps_t in [0.5, 1.0, 3.0, 10.0]:
    _, a, u, s = dp_sgd(X_train, y_train, X_test, y_test, eps=eps_t, epochs=20, bs=64, lr=5e-3)
    dp_rows.append({"epsilon": f"{eps_t}", "Acc (%)": f"{a*100:.2f}", "AUC": f"{u:.4f}", "sigma": f"{s:.3f}"})
df_dp = pd.DataFrame(dp_rows)
print(df_dp.to_string(index=False))
df_dp.to_csv(f"{OUT}/batch6_dp.csv", index=False)
def temporal_drift(model, X, y, T=50):
    torch.manual_seed(SEED)
    accs, snrs = [], []
    for t in range(T+1):
        snr = 30 - 25 * (t/T); snrs.append(snr)
        atten = np.exp(-0.02 * t/T)
        Xt = torch.tensor(X).float() * atten
        Xt = add_noise(Xt, snr).numpy()
        accs.append(evaluate(model, Xt, y)[0])
    return snrs, accs
snrs_t, accs_ndp = temporal_drift(warm_net, X_test, y_test, 50)
df_drift6 = pd.DataFrame({"T": list(range(51)), "SNR": [f"{s:.2f}" for s in snrs_t],
                          "Acc (%)": [f"{a*100:.2f}" for a in accs_ndp]})
df_drift6.to_csv(f"{OUT}/batch6_drift.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
eps_vals = [10, 3, 1, 0.5]
acc_vals = [float(dp_rows[1]["Acc (%)"]), float(dp_rows[2]["Acc (%)"]),
            float(dp_rows[3]["Acc (%)"]), float(dp_rows[4]["Acc (%)"])]
ax[0].plot(eps_vals, acc_vals, 'o-', color='steelblue', linewidth=2)
ax[0].axhline(base_acc_6*100, ls='--', color='red', alpha=0.6, label=f'Non-DP ({base_acc_6*100:.2f}%)')
ax[0].set_xscale('log'); ax[0].set_xlabel("epsilon"); ax[0].set_ylabel("Acc (%)")
ax[0].set_title("(a) DP-SGD trade-off"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(range(51), [a*100 for a in accs_ndp], 'o-', color='steelblue', label='Non-DP')
ax[1].axhline(79.48, ls=':', color='grey', alpha=0.5, label='Majority')
ax[1].set_xlabel("Timestep"); ax[1].set_ylabel("Acc (%)")
ax[1].set_title("(b) Temporal drift"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT}/batch6_figure.png", dpi=200); plt.close()
display(Image(filename=f"{OUT}/batch6_figure.png"))
print("\n=== Batch 7 ===")
def detect_s1s2(wav):
    env = np.abs(wav)
    win = int(0.02 * SR)
    sm = np.convolve(env, np.ones(win)/win, 'same')
    peaks, _ = find_peaks(sm, distance=int(0.15*SR),
                          height=np.percentile(sm, 75))
    h = int(0.075 * SR)
    return [(max(0, p-h), min(len(wav), p+h)) for p in peaks if 2*h >= int(0.1*SR)]
def build_s1s2(wavs, desc):
    X, Y = [], []
    for wp, lb in tqdm(wavs, desc=desc, leave=False):
        wav = load_wave(wp); segs = detect_s1s2(wav)
        if not segs: mel = wav_to_mel(wav)
        else:
            best = max(segs, key=lambda s: np.sum(wav[s[0]:s[1]]**2))
            seg = wav[best[0]:best[1]]
            tgt = int(1.0*SR)
            seg = seg[:tgt] if len(seg) >= tgt else np.pad(seg, (0, tgt-len(seg)))
            mel = wav_to_mel(seg)
        if mel.shape[1] < 157: mel = np.pad(mel, ((0,0),(0,157-mel.shape[1])))
        X.append(mel[:, :157]); Y.append(lb)
    return np.array(X)[:,None,:,:].astype(np.float32), np.array(Y).astype(np.int64)
if os.path.isfile(CACHE_S1S2):
    _s = np.load(CACHE_S1S2)
    X_train_s, y_train_s, X_test_s, y_test_s = _s["X_train"], _s["y_train"], _s["X_test"], _s["y_test"]
else:
    X_train_s, y_train_s = build_s1s2(tr_list, "s1s2 train")
    X_test_s, y_test_s = build_s1s2(te_list, "s1s2 test")
    np.savez_compressed(CACHE_S1S2, X_train=X_train_s, y_train=y_train_s,
                        X_test=X_test_s, y_test=y_test_s)
net_2s = warm_net
acc_2s, auc_2s = evaluate(net_2s, X_test, y_test)
net_s1s2, _ = train_warm(X_train_s, y_train_s, X_test_s, y_test_s, epochs=40)
acc_s1s2, auc_s1s2 = evaluate(net_s1s2, X_test_s, y_test_s)
df_c8 = pd.DataFrame([
    {"Input": "2s full clip", "Acc (%)": f"{acc_2s*100:.2f}", "AUC": f"{auc_2s:.4f}"},
    {"Input": "S1/S2 single-cycle", "Acc (%)": f"{acc_s1s2*100:.2f}", "AUC": f"{auc_s1s2:.4f}"},
])
print(df_c8.to_string(index=False))
df_c8.to_csv(f"{OUT}/batch7_s1s2.csv", index=False)
rng = np.random.RandomState(SEED)
abn = np.where(y_train == 1)[0]; nor = np.where(y_train == 0)[0]
rng.shuffle(abn)
n_unseen = int(0.3 * len(abn))
seen = abn[n_unseen:]; unseen = abn[:n_unseen]
X_os = X_train[np.concatenate([nor, seen])]; y_os = y_train[np.concatenate([nor, seen])]
X_te_os = np.concatenate([X_test, X_train[unseen]], 0)
y_te_os = np.concatenate([y_test, y_train[unseen]], 0)
net_os, _ = train_warm(X_os, y_os, X_te_os, y_te_os, epochs=40)
acc_os, auc_os = evaluate(net_os, X_te_os, y_te_os)
df_c17 = pd.DataFrame([
    {"Setting": "Closed-set", "Acc (%)": f"{acc_2s*100:.2f}", "AUC": f"{auc_2s:.4f}"},
    {"Setting": "Open-set (30% unseen)", "Acc (%)": f"{acc_os*100:.2f}", "AUC": f"{auc_os:.4f}"},
])
print(df_c17.to_string(index=False))
df_c17.to_csv(f"{OUT}/batch7_open_set.csv", index=False)
def identify_murmur_regions(mel, smooth_win=25, quantile=0.75, min_len=8):
    band = mel[2:9, :]
    energy = band.mean(axis=0)
    kernel = np.ones(smooth_win) / smooth_win
    smooth = np.convolve(energy, kernel, mode='same')
    thr = np.quantile(smooth, quantile)
    mask = smooth > thr
    regions = []
    start = None
    for i, m in enumerate(mask):
        if m and start is None:
            start = i
        elif not m and start is not None:
            if i - start >= min_len:
                regions.append((start, i))
            start = None
    if start is not None and len(mask) - start >= min_len:
        regions.append((start, len(mask)))
    return regions
n_reg_normal, n_reg_abn = [], []
for wp, lb in tqdm(te_list[:200], desc="murmur", leave=False):
    wav = load_wave(wp); mel = wav_to_mel(wav)
    regions = identify_murmur_regions(mel)
    if lb == 0: n_reg_normal.append(len(regions))
    else: n_reg_abn.append(len(regions))
df_c19 = pd.DataFrame([
    {"Class": "Normal", "Mean murmur regions": f"{np.mean(n_reg_normal):.2f} +/- {np.std(n_reg_normal):.2f}"},
    {"Class": "Abnormal", "Mean murmur regions": f"{np.mean(n_reg_abn):.2f} +/- {np.std(n_reg_abn):.2f}"},
])
print(df_c19.to_string(index=False))
df_c19.to_csv(f"{OUT}/batch7_murmur.csv", index=False)
test_wavs_lookup = {os.path.basename(wp): (wp, lb) for wp, lb in te_list}
test_names = [os.path.basename(wp) for wp, _ in te_list]
normal_samples = [n for n in test_names if test_wavs_lookup[n][1] == 0][:2]
abn_samples = [n for n in test_names if test_wavs_lookup[n][1] == 1][:2]
show_samples = normal_samples + abn_samples
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, name in enumerate(show_samples):
    wp, lb = test_wavs_lookup[name]
    wav = load_wave(wp); mel = wav_to_mel(wav)
    ax = axes[0, i]
    ax.imshow(mel, aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(f"{'Normal' if lb==0 else 'Abnormal'}\n{name[:20]}", fontsize=9)
    ax.set_ylabel("Mel bin" if i == 0 else ""); ax.set_xticks([])
    ax = axes[1, i]
    regions = identify_murmur_regions(mel)
    ax.imshow(mel, aspect='auto', origin='lower', cmap='viridis')
    for (s, e) in regions:
        ax.axvspan(s, e, color='red', alpha=0.25)
    ax.set_xlabel("Time frame", fontsize=9)
    ax.set_title(f"{len(regions)} candidate region(s)", fontsize=9)
    ax.set_ylabel("Mel bin" if i == 0 else "")
plt.suptitle("Candidate murmur regions overlayed on log-Mel spectrograms", fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(f"{OUT}/batch7_murmur_overlay.png", dpi=200, bbox_inches='tight'); plt.close()
display(Image(filename=f"{OUT}/batch7_murmur_overlay.png"))
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(2); w = 0.35
ax[0].bar(x-w/2, [acc_2s*100, acc_s1s2*100], w, label='Accuracy', color='steelblue')
ax[0].bar(x+w/2, [auc_2s*100, auc_s1s2*100], w, label='AUC x 100', color='salmon')
ax[0].set_xticks(x); ax[0].set_xticklabels(["2s","S1/S2"]); ax[0].legend()
ax[0].set_ylabel("Value (%)"); ax[0].set_title("(a) S1/S2 segmentation"); ax[0].grid(axis='y', alpha=0.3)
ax[1].bar(x-w/2, [acc_2s*100, acc_os*100], w, label='Accuracy', color='steelblue')
ax[1].bar(x+w/2, [auc_2s*100, auc_os*100], w, label='AUC x 100', color='salmon')
ax[1].set_xticks(x); ax[1].set_xticklabels(["Closed","Open"]); ax[1].legend()
ax[1].set_ylabel("Value (%)"); ax[1].set_title("(b) Open-set simulation"); ax[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT}/batch7_figure.png", dpi=200); plt.close()
display(Image(filename=f"{OUT}/batch7_figure.png"))
zp = shutil.make_archive("all_batches_results", "zip", OUT)
print(f"\nPackaged: {zp}")
try:
    from google.colab import files
    files.download(zp)
except Exception:
    print(f"Manual: {zp}")